In [2]:
import numpy as np
import matplotlib.pyplot as plt
import time as time
import re
import os
import sys


# go one folder up from the current working directory
parent_dir = os.path.abspath("..")

# build the path to folder_b
path = os.path.join(parent_dir)

# add it to sys.path
sys.path.append(path)

import vector_borne_functions as vbf

T = vbf.T
f = vbf.f
gammas = vbf.gammas
g = vbf.g
chis = vbf.chis
F = vbf.F
daily_mean_arrays = vbf.daily_mean_arrays
dX_ms = vbf.dX_ms
Vector_borne_ms = vbf.Vector_borne_ms
dX_Rinf_ms = vbf.dX_Rinf_ms
#Vector_borne_Rinf_ms = vbf.Vector_borne_Rinf_ms
Vector_borne_Rinf_noise = vbf.Vector_borne_Rinf_noise
plot_compartments_vs_t = vbf.plot_compartments_vs_t



def get_climatic_arrays_sigma(folder_name,day_fraction,sigma):

    dimension, a_min, a_max,b_min,b_max = [int(x) for x in re.findall(r"[-+]?\d*\.?\d+", folder_name)][:5]

    # a and b values arrays
    a_values = np.linspace(a_min,a_max,dimension)
    b_values = np.linspace(b_min,b_max,dimension)

    # factor to work in years 
    factor = 365

    mean_f_t_array = np.zeros((dimension,dimension))
    mean_g_t_array = np.zeros((dimension,dimension))

    times = times = np.array(range(365*day_fraction))/(365*day_fraction)

    for row in range(dimension):
        #print(row)
            for column in range(dimension):
                            
                a,b = a_values[row], b_values[column]

                if b>12 and b>a:

                    T_t = T(times,a,b) + np.sqrt(sigma) * np.random.normal(0, 1, 365*day_fraction)
                    
                    f_t = f(T_t)*365
                    g_t = g(T_t)*365

                    """
                    f_t = daily_mean_arrays(f_t,day_fraction)*factor
                    g_t = daily_mean_arrays(g_t,day_fraction)*factor
                    """
                
                    f_t_mean = np.mean(f_t[:day_fraction*365])
                    g_t_mean = np.mean(g_t[:day_fraction*365])
                        
                    mean_f_t_array[row,column] = f_t_mean      
                    mean_g_t_array[row,column] = g_t_mean
                
                if b<a or b<12:

                    mean_f_t_array[row,column] = -0.1
                    mean_g_t_array[row,column] = -0.1

    return mean_f_t_array, mean_g_t_array



def sum_alpha_i(MGDD_R):
    

    c_1 = 0.012
    c_2 = 975
    
    sum_alpha = MGDD_R + 1/c_1*np.log( (1 + np.exp(-c_1 * (MGDD_R-c_2) ) ) /(1 + np.exp(c_1 * c_2) ) ) 

    return sum_alpha


def sum_g(n,mean_g_t_array, mean_f_t_array):

    MGDD_R = 1500.
    Delta_MGDD=MGDD_R/n
    MGDD = np.array([x*Delta_MGDD for x in range(n+1)])

    s=0
    for i in range(1,n-1):
        s_=0
        for j in range(i-1,n-2):
        
            s_+= (mean_g_t_array/mean_f_t_array)**(n-(2+j))

        s+=F(MGDD[i])*s_

    return s*Delta_MGDD
    



def get_r_0(f_m_array, g_m_array,sum_g_f, Alpha, beta, mu, Gamma):

    MGDD_R = 1500.
    alpha_sum = sum_alpha_i(MGDD_R)

    R_0_aproximation_array = Alpha * beta / (mu * Gamma) * ( 1 + Gamma *( alpha_sum + sum_g_f ) / f_m_array )  * (1-(g_m_array / f_m_array)) #/(1-(mean_g_t_array / mean_f_t_array)**(n-1))
    R_0_aproximation_array [R_0_aproximation_array <0] = 0


    mask = (f_m_array<0) & (g_m_array<0) 
    R_0_aproximation_array[mask] = -0.1

    return R_0_aproximation_array

In [3]:
#list to store the failed jobs (simulation data that should be in R_folders and do not)
failed_jobs = []


# path to the directory with the data of R_{\infty}}
cwd = os.getcwd()

#parent = os.path.dirname(cwd) + "/adding_noise"

parent_dir = cwd  + '/R_infty_arrays'


# list of folder in the folder "R_0_arrays"
R_folders = os.listdir(parent_dir)
#chosing one folder
#folder_name = input(f'las carpetas que hay son {R_folders} elige una:')
folder_name = R_folders[0]

# path to the folder with data of the simulations with "dimension", "a_min", ...
directory = parent_dir + '/' + folder_name

# names of the arrays in the folder with data of the simulations with "dimension", "a_min", ...
# these arrays have names Arr_{Alpha}_{beta}_{mu}_{Gamma}

folder_arrays = os.listdir(directory)

# parameters of the simulation that give rise to the data in each array
disordered_parameters = [[float(x) for x in re.findall(r"[-+]?\d*\.?\d+", y)] for y in folder_arrays if re.search(r'\d', y)]


disordered_parameters = np.array(disordered_parameters)

ordered_parameters = np.loadtxt(directory + '/params.txt')

parameters = np.array([row for row in ordered_parameters  if  any((row == disordered_parameters).all(axis=1))])

# list to store the arrays 
R_inf_arrays = []


for parameter_combination in parameters:
    
    Alpha, sigma, realization = parameter_combination
    
    R_inf_arrays.append(np.loadtxt(directory + f'/R_inf_values_{Alpha}_{sigma}_{realization}.txt'))

In [4]:
# path to the directory with the data of R_{\infty}}
#cwd = os.getcwd()


sigma_0_folder = cwd  + '/R_infty_arrays' + '/dimension_70_arage_-20_26_brange_12_50_dfraction_5_n200_Iv_noise_sigma_0'

R_inf_sigma_0_arrays = []

parameters_sigma_0 = np.loadtxt(sigma_0_folder + '/params.txt')


for parameter_combination in parameters_sigma_0:

    Alpha, sigma, realization = parameter_combination

    R_inf_sigma_0_arrays.append(np.loadtxt(sigma_0_folder+ f'/R_inf_values_{Alpha}_{sigma}_{realization}.txt'))

In [5]:
folder_name = 'dimension_70_arage_-20_26_brange_12_50_dfraction_5_n200_Iv_noise'
dimension, a_min, a_max,b_min,b_max = [int(x) for x in re.findall(r"[-+]?\d*\.?\d+", folder_name)][:5]

day_fraction = re.search(r"dfraction_(\d+)", folder_name)
n = re.search(r"n(\d+)", folder_name)

day_fraction = int(day_fraction.group(1)) if day_fraction else None
n = int(n.group(1)) if n else None


day_fraction = 3
sigma = 20.25

In [6]:
n_realizations = 100


f_m_m_array = np.zeros((dimension,dimension),float)
g_m_m_array = np.zeros((dimension,dimension),float)
r_0_m_array = np.zeros((dimension,dimension),float)




for k in range(n_realizations):
    print(k)
    f_m_array,g_m_array = get_climatic_arrays_sigma(folder_name,day_fraction,sigma)

    f_m_m_array += f_m_array
    g_m_m_array += g_m_array

    sum_g_f = sum_g(n,g_m_array, f_m_array)

    r_0_m_array += get_r_0(f_m_array, g_m_array,sum_g_f, 1, 1, 1, 1)




f_m_0,g_m_0 = get_climatic_arrays_sigma(folder_name,day_fraction,0)
sum_g_f = sum_g(n,g_m_0, f_m_0)
r_0_0 = get_r_0(f_m_0, g_m_0,sum_g_f, 1, 1, 1, 1)

f_m_m_array /= n_realizations
g_m_m_array /= n_realizations
r_0_m_array /= n_realizations


delta_f_m = f_m_m_array - f_m_0
delta_g_m = g_m_m_array - g_m_0
delta_r_0_m = r_0_m_array - r_0_0


np.savetxt('delta_f_m.txt', delta_f_m)
np.savetxt('delta_g_m.txt', delta_g_m)
np.savetxt('delta_r_0_m.txt', delta_r_0_m)

0


/tmp/ipykernel_1706257/1255962144.py:107: RuntimeWarning: invalid value encountered in divide
  s_+= (mean_g_t_array/mean_f_t_array)**(n-(2+j))
/tmp/ipykernel_1706257/1255962144.py:107: RuntimeWarning: overflow encountered in power
  s_+= (mean_g_t_array/mean_f_t_array)**(n-(2+j))
/tmp/ipykernel_1706257/1255962144.py:107: RuntimeWarning: invalid value encountered in divide
  s_+= (mean_g_t_array/mean_f_t_array)**(n-(2+j))
/tmp/ipykernel_1706257/1255962144.py:107: RuntimeWarning: overflow encountered in power
  s_+= (mean_g_t_array/mean_f_t_array)**(n-(2+j))
/tmp/ipykernel_1706257/1255962144.py:121: RuntimeWarning: invalid value encountered in divide
  R_0_aproximation_array = Alpha * beta / (mu * Gamma) * ( 1 + Gamma *( alpha_sum + sum_g_f ) / f_m_array )  * (1-(g_m_array / f_m_array)) #/(1-(mean_g_t_array / mean_f_t_array)**(n-1))


1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24


/tmp/ipykernel_1706257/1255962144.py:107: RuntimeWarning: overflow encountered in add
  s_+= (mean_g_t_array/mean_f_t_array)**(n-(2+j))


25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
